In [4]:
import re
import pandas as pd

from google_sheets import GoogleSheetsAPI
from legiscan import LegiscanClient

In [7]:
sheets = GoogleSheetsAPI()
try:
    df = sheets.read_sheet()
except ValueError as e:
    print(e)
client = LegiscanClient()
df = df[df["Level of Government"] == "State"]
df = df.dropna(how="all")
df["Introduction Date"] = pd.to_datetime(df["Introduction Date"], format="mixed", errors="coerce")

In [8]:
rows = df.to_dict("records")  # type: ignore
for row in rows:
    state = row["Jurisdiction"]
    # if state not in LegiscanClient.STATE_NAME_TO_ABBR:
    #     print(state)
    state = LegiscanClient.STATE_NAME_TO_ABBR[state]
    queries = [
        row["Bill Number"].replace(" ", ""),
        re.sub(r"[A-Za-z]", "", row["Bill Number"]),
    ]
    result, query = None, None
    try:
        for query in queries:
            result = client.get_search(state, query)
            if result["status"] == "OK" and result["searchresult"]["summary"]["count"] > 0:
                break
        if (result is None) or (result and result["searchresult"]["summary"]["count"] == 0):
            print(f"No bill found for ({state}, {query}): {row['Bill Number']}")
    except Exception as e:
        print(f"Failed to search for ({state}, {query}): - {row['Bill Number']}")
        print(e)


No bill found for (ID, 0492): H0492
No bill found for (IN, 0445): SB0445
No bill found for (IN, 0161): SB0161
No bill found for (NY, 07135): S07135
No bill found for (NY, 03593): A03593
No bill found for (NY, 00365): S00365
No bill found for (NY, 07423): A07423
No bill found for (NY, 03308): A03308
No bill found for (NY, 03226): S03226
No bill found for (NY, 05251): A05251
No bill found for (NY, 04162): S04162
No bill found for (NY, 00987): S00987
No bill found for (NY, 05686): A05686
No bill found for (NY, 01877): A01877
No bill found for (NY, 04638): A04638
No bill found for (NY, 00998): S00998
No bill found for (NY, 01764): S01764
No bill found for (NY, 03306): A03306
No bill found for (NY, 00217): S00217
No bill found for (NY, 03311): A03311
No bill found for (NY, 01891): A01891
No bill found for (NY, 01609): S01609
Failed to search for (TX,      11..503.001): - Texas Business and Commerce Code 11.A.503.001
'searchresult'
No bill found for (VT, 0121): H0121
No bill found for (VA,  